In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Portfolio Optimizer: A Qiskit Function by Global Data Quantum



> **Note:** Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (via IBM Quantum Platform API) Plan. Fitur ini dalam status rilis pratinjau dan dapat berubah sewaktu-waktu.
# Gambaran Umum
Quantum Portfolio Optimizer adalah sebuah Qiskit Function yang menangani masalah optimasi portofolio dinamis, sebuah masalah standar dalam keuangan yang bertujuan untuk menyeimbangkan kembali investasi periodik di sejumlah aset, untuk memaksimalkan keuntungan dan meminimalkan risiko. Dengan menggunakan teknik optimasi kuantum mutakhir, fungsi ini menyederhanakan prosesnya sehingga pengguna, tanpa keahlian di bidang komputasi kuantum, bisa memanfaatkan keunggulannya dalam menemukan trajektori investasi yang optimal. Ideal untuk manajer portofolio, peneliti keuangan kuantitatif, dan investor individu, alat ini memungkinkan back-testing strategi trading dalam optimasi portofolio.
# Deskripsi Fungsi
Fungsi Quantum Portfolio Optimizer menggunakan algoritma Variational Quantum Eigensolver (VQE) untuk menyelesaikan masalah Quadratic Unconstrained Binary Optimization (QUBO), yang menangani masalah optimasi portofolio dinamis. Pengguna cukup menyediakan data harga aset dan mendefinisikan batasan investasi, lalu fungsi ini menjalankan proses optimasi kuantum yang mengembalikan sekumpulan trajektori investasi yang optimal.

Proses ini terdiri dari empat tahap utama. Pertama, data input dipetakan ke masalah yang kompatibel dengan kuantum, membangun QUBO dari masalah optimasi portofolio dinamis, dan mengubahnya menjadi operator kuantum (Ising Hamiltonian). Selanjutnya, masalah input dan algoritma VQE diadaptasi agar dapat dijalankan di perangkat keras kuantum. Algoritma VQE kemudian dijalankan di perangkat keras kuantum, dan akhirnya, hasilnya diproses pasca-komputasi untuk memberikan trajektori investasi yang optimal. Sistem ini juga mencakup pasca-pemrosesan yang peka terhadap noise (berbasis [SQD](/guides/qiskit-addons-sqd)) untuk memaksimalkan kualitas output.

Qiskit Function ini didasarkan pada [naskah yang sudah dipublikasikan](https://arxiv.org/abs/2412.19150) oleh Global Data Quantum.
![Visualisasi alur kerja fungsi](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Input
Argumen input dari fungsi ini dijelaskan dalam tabel berikut. Data aset dan spesifikasi masalah lainnya harus disediakan; selain itu, pengaturan VQE bisa dimasukkan untuk menyesuaikan proses optimasi.
| **Nama**               | **Tipe**           | **Deskripsi**                                          | **Wajib** | **Default**                          | **Contoh**                              |
|------------------------|--------------------|----------------------------------------------------------|--------------|--------------------------------------|------------------------------------------|
| assets                 | json               | Dictionary berisi harga aset                         | Ya          | -                                    | -                                        |
| qubo_settings          | json               | Pengaturan QUBO                                     | Ya          | -                                    | Lihat contoh di tabel di bawah     |
| ansatz_settings        | json               | Pengaturan ansatz                                   | Tidak           | `None`                                | Lihat contoh di tabel di bawah.     |
| optimizer_settings     | json               | Pengaturan optimizer                                | Tidak           | `None`                                | Lihat contoh di tabel di bawah.     |
| backend                | str                | Nama Backend QPU                                     | Tidak           |  -    | `"ibm_torino"`                           |
| previous_session_id    | list of str        | Daftar ID sesi untuk mengambil data dari eksekusi sebelumnya[(*)](#1-note) | Tidak           | List kosong                           | `["session_id_1", "session_id_2"]`      |
| apply_postprocess      | bool               | Terapkan pasca-pemrosesan SQD yang peka terhadap noise                      | Tidak           | `True`                                | `True`                                   |
| tags                   | list of strings    | Daftar tag untuk mengidentifikasi eksperimen                  | Tidak           | List kosong                           | `["optimization", "quantum_computing"]`  |
<span id="1-note"></span>
*Untuk melanjutkan eksekusi atau mengambil job yang diproses dalam satu atau lebih sesi sebelumnya, daftar ID sesi harus diteruskan dalam parameter `previous_session_id`. Ini sangat berguna dalam kasus di mana tugas optimasi gagal diselesaikan karena kesalahan dalam proses, dan eksekusi perlu diselesaikan. Untuk mencapai ini, kamu harus memberikan argumen yang sama yang digunakan dalam eksekusi awal, beserta daftar `previous_session_id` seperti yang dijelaskan.
> **Caution:** Memuat data untuk sesi sebelumnya (untuk melanjutkan optimasi) bisa membutuhkan waktu hingga satu jam.
## `assets`
Data harus disusun sebagai objek JSON yang menyimpan informasi tentang harga penutupan aset keuangan pada tanggal tertentu. Formatnya adalah sebagai berikut:

- Kunci primer (string): Nama atau simbol ticker aset keuangan (misalnya, "8801.T").
- Kunci sekunder (string): Tanggal dalam format YYYY-MM-DD.
- Nilai (number): Harga penutupan aset pada tanggal yang ditentukan. Harga bisa dimasukkan dalam bentuk normalisasi atau tidak.

*Perhatikan bahwa semua dictionary harus memiliki kunci sekunder yang sama (tanggal). Jika suatu aset tidak memiliki tanggal yang dimiliki aset lainnya, data harus diisi untuk memastikan konsistensi. Misalnya, ini bisa dilakukan dengan menggunakan harga penutupan terakhir yang tercatat dari aset tersebut.*
### Contoh
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** Data aset harus mengandung, setidaknya, harga penutupan pada ``(nt+1) * dt`` (lihat bagian input [`qubo_settings`](#qubo_settings)) time stamp (misalnya, hari).
## `qubo_settings`
Tabel berikut menjelaskan kunci dari dictionary `qubo_settings`. Buat dictionary dengan menentukan jumlah langkah waktu `nt`, jumlah qubit resolusi `nq`, dan `max_investment` - atau ubah nilai default lainnya.
| Nama                | Tipe    | Deskripsi                                                                 | Wajib | Default | Contoh |
|---------------------|---------|-----------------------------------------------------------------------------|----------|---------|---------|
| nt                  | int     | Jumlah langkah waktu                                                         | Ya      | -       | 4       |
| nq                  | int     | Jumlah Qubit resolusi                                                                | Ya      | -       | 4       |
| max_investment      | float   | Jumlah maksimum unit mata uang yang diinvestasikan di semua aset                           | Ya      | -       | 10      |
| dt*                  | int     | Jendela waktu yang dipertimbangkan di setiap langkah waktu. Satuannya sesuai dengan interval waktu antara kunci dalam data aset                                                 | Tidak       | 30      | -       |
| risk_aversion       | float     | Koefisien aversion risiko                                   | Tidak       | 1000    | -       |
| transaction_fee     | float     | Koefisien biaya transaksi                                                 | Tidak       | 0.01   | -       |
| restriction_coeff   | float   | Pengali Lagrange yang digunakan untuk memaksakan batasan masalah dalam formulasi QUBO | Tidak       | 1       | -       |
## `ansatz_settings`
Untuk mengubah opsi default, buat dictionary untuk parameter `ansatz_settings` dengan kunci berikut. Secara default, ansatz diatur ke `"real_amplitudes"`, dan kedua opsi tambahan (lihat tabel berikut) diatur ke `False`.
| Nama                  | Tipe     | Deskripsi                                                                 | Wajib | Default |
|-----------------------|----------|-----------------------------------------------------------------------------|----------|---------|
| ansatz[*](#single-star)                | str      | Ansatz yang akan digunakan                                             | Tidak      | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star)  | bool     |Mengaktifkan subrutin multiple passmanager (tidak tersedia untuk Tailored ansatz) | Tidak       | `False`   |
| dd_enable   | bool     | Menambahkan dynamical decoupling                                    | Tidak       | `False`   |

<span id="single-star"></span>
\* Ansatz yang tersedia
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored` (Hanya untuk Backend `ibm_torino`, 7 aset, 4 langkah waktu, dan 4 Qubit resolusi)

<span id="double-star"></span>
** Jika ``multiple_passmanager`` diatur ke ``False``, fungsi menggunakan pass manager Qiskit default dengan `optimization_level=3`. Jika diatur ke ``True``, subrutin ``multiple_passmanager`` membandingkan tiga pass manager: pass manager Qiskit default sebelumnya, pass manager yang memetakan Qubit di atas rantai tetangga pertama QPU, dan layanan [AI Transpiler](/guides/ai-transpiler-passes). Kemudian, pass manager dengan estimasi error kumulatif terendah dipilih.
## `optimizer_settings`
Parameter ini adalah dictionary dengan beberapa opsi yang bisa disesuaikan dari proses optimasi.
| Nama               | Tipe   | Deskripsi                                     | Wajib | Default               |
|--------------------|--------|-------------------------------------------------|----------|-----------------------|
| primitive_options  | json   | Pengaturan primitive               | Tidak      | -                     |
| optimizer          | str    | Optimizer klasik yang dipilih                            | Tidak       | `"differential_evolution"` |
| optimizer_options  | json   | Konfigurasi optimizer                  | Tidak       | -                     |
> **Note:** Saat ini, satu-satunya opsi optimizer yang tersedia adalah `"differential_evolution"`.

Di bawah kunci `primitive_options` dan `optimizer_options`, kita menetapkan dictionary dengan parameter berikut:
### `primitive_options`
| Nama              | Tipe   | Deskripsi                                | Wajib | Default                    | Contoh                    |
|-------------------|--------|--------------------------------------------|----------|----------------------------|----------------------------|
| sampler_shots     | int    | Jumlah shots dari Sampler.            | Tidak       | 100000                     | -                          |
| estimator_shots   | int    | Jumlah shots dari Estimator.          | Tidak       | 25000                      | -                          |
| estimator_precision | float | Presisi yang diinginkan dari nilai yang diharapkan. Jika ditentukan, presisi akan digunakan sebagai pengganti `estimator_shots`. | Tidak | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int or str | Jumlah waktu maksimum yang bisa dibuka oleh sesi runtime sebelum ditutup secara paksa. Bisa diberikan dalam detik (int) atau sebagai string, seperti `"2h 30m 40s"`. Harus kurang dari maksimum yang ditentukan sistem. | Tidak | `None` | `"1h 15m"`                |
### `optimizer_options`
| Nama              | Tipe     | Deskripsi                              | Wajib | Default       |
|-------------------|----------|------------------------------------------|----------|---------------|
| num_generations   | int      | Jumlah generasi                    | Tidak       | 20            |
| population_size   | int      | Ukuran populasi                    | Tidak       | 20            |
| mutation_range    | list   | Faktor mutasi maksimum dan minimum              | Tidak       | [0, 0.25]     |
| recombination     | float      | Faktor rekombinasi                     | Tidak       | 0.4           |
| max_parallel_jobs | int      | Jumlah maksimum job QPU yang dieksekusi secara paralel  | Tidak       | 3             |
| max_batchsize | int      | Ukuran batch maksimum  | Tidak       | 200     |
> **Note:** - Jumlah generasi yang dievaluasi oleh differential evolution adalah `num_generations` + 1 karena populasi awal sudah termasuk.
> - Total jumlah Circuit dihitung sebagai `(num_generations + 1) * population_size`.
> - Menggunakan ukuran populasi yang lebih besar dan lebih banyak generasi umumnya meningkatkan kualitas hasil optimasi. Namun, tidak disarankan untuk melebihi ukuran populasi 120 dan jumlah generasi lebih dari 20 (misalnya, ``120 * 21 = 2520`` total Circuit), karena ini akan menghasilkan jumlah Circuit yang berlebihan, yang bisa mahal secara komputasi dan memakan waktu untuk diproses.
> 
> - Fungsi ini memungkinkan kamu untuk melanjutkan optimasi sebelumnya, dan selalu mungkin untuk menambah jumlah generasi (dengan memberikan input yang sama kecuali untuk `previous_session_id` dan ``num_generations`` yang ditingkatkan).
> **Note:** Pastikan kepatuhan dengan batas job Qiskit Runtime.
> 
> - Sampler: `sampler_shots <= 10_000_000`.
> - Estimator: `max_batchsize * estimator_shots * observable_size <= 10_000_000` (untuk fungsi ini, semua istilah observable saling komute, sehingga `observable_size=1`).
> 
> Lihat panduan [Batas Job](/guides/job-limits) untuk informasi lebih lanjut.
# Output
Fungsi ini mengembalikan dua dictionary: dictionary `"result"`, yang berisi hasil optimasi terbaik, termasuk solusi optimal dan biaya objektif minimum yang terkait; dan `"metadata"`, dengan data dari semua hasil yang diperoleh selama proses optimasi, beserta metrik masing-masing.

Dictionary pertama berfokus pada solusi dengan performa terbaik, sementara yang kedua memberikan informasi detail tentang semua solusi, termasuk biaya objektif dan metrik relevan lainnya.

## Dictionary Output:
| **Nama**    | **Tipe**                     | **Deskripsi**                                                                 | **Contoh**  |
|-------------|------------------------------|---------------------------------------------------------------------------------|--------------|
| result      | dict[str, dict[str, float]]  | Berisi strategi investasi dari waktu ke waktu, dengan setiap time-stamp dipetakan ke bobot investasi spesifik aset (setiap bobot adalah jumlah investasi yang dinormalisasi dengan total jumlah investasi). | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | Data yang dihasilkan selama analisis, termasuk solusi, biaya, dan metrik.    | Lihat contoh di bawah |
### Deskripsi dictionary `metadata`
| **Nama**                 | **Tipe**              | **Deskripsi**                                                                                     | **Contoh**                  |
|--------------------------|-----------------------|-----------------------------------------------------------------------------------------------------|------------------------------|
| session_id               | str                   | Pengenal unik untuk sesi IBM Quantum.                          | `"d0h30qjvpqf00084fgw0"`        |
| all_samples_metrics | dict                 | Dictionary yang berisi berbagai metrik untuk setiap sampel yang telah diproses pasca-komputasi, seperti biaya atau batasan. | Lihat deskripsi di bawah        |
| sampler_counts           | dict[str, int]        | Dictionary di mana kunci adalah representasi bitstring dari solusi yang disampel dan nilainya adalah jumlah kemunculannya. | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | Daftar dengan urutan investasi aset yang sesuai di setiap langkah waktu dalam strategi investasi. | `["Asset_0", "Asset_1", "Asset_3"]`     |
| QUBO                     | list[list[float]]     | Matriks QUBO dari masalah.                                                                         | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]`     |
| resource_summary           | dict[str, dict[str, float]] | Ringkasan waktu penggunaan CPU dan QPU (dalam detik) di berbagai tahap proses.                | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |
### Deskripsi dictionary `all_samples_metrics`
| **Nama**                | **Tipe**       | **Deskripsi**                                                                                                  | **Contoh**                  |
|-------------------------|----------------|------------------------------------------------------------------------------------------------------------------|------------------------------|
| investment_trajectories | list[list]     | Strategi investasi yang diturunkan dari state kuantum yang didekodekan. | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | Jumlah berapa kali setiap trajektori investasi disampel. Indeks sesuai dengan `investment_trajectories`.                | `[5, 3]`                     |
| objective_costs         | list[float]    | Nilai fungsi objektif untuk setiap trajektori investasi, diurutkan dari terendah ke tertinggi.                 | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | Performa yang disesuaikan risiko (Sharpe ratio) untuk setiap trajektori investasi. Selaras berdasarkan indeks.                      | `[1.1, 0.7]`                 |
| returns                 | list[float]    | Return yang diharapkan untuk setiap trajektori investasi. Selaras berdasarkan indeks.                                               | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | Deviasi batasan maksimum dalam setiap trajektori investasi. Selaras berdasarkan indeks.                               | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | Perkiraan biaya transaksi yang terkait dengan setiap trajektori investasi. Selaras berdasarkan indeks.                        | `[0.01, 0.02]`               |
# Mulai
Autentikasi menggunakan [kunci API](https://quantum.cloud.ibm.com/) kamu dan pilih Qiskit Function sebagai berikut. (Cuplikan ini mengasumsikan kamu sudah [menyimpan akunmu](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokal.)

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## Contoh: Optimasi portofolio dinamis dengan tujuh aset
Contoh ini menunjukkan cara mengeksekusi fungsi dynamic portfolio optimization (DPO) dan menyesuaikan pengaturannya untuk performa optimal. Ini mencakup langkah-langkah detail untuk menyetel parameter agar mencapai hasil yang diinginkan.

Kasus ini melibatkan tujuh aset, empat langkah waktu, dan empat Qubit resolusi, sehingga menghasilkan total kebutuhan 112 Qubit.
### 1. Baca aset yang termasuk dalam portofolio.
Jika semua aset dalam portofolio disimpan di folder pada path tertentu, kamu bisa memuatnya ke dalam `pandas.DataFrame` dan mengonversinya ke objek format `dict` menggunakan fungsi berikut.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

Untuk contoh ini, kami menggunakan aset [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y), dan [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Gambar berikut mengilustrasikan data yang digunakan dalam contoh ini, menunjukkan evolusi harga penutupan harian aset dari 1 Januari hingga 1 September 2023.

Dalam contoh ini, untuk memastikan keseragaman antar tanggal, kami mengisi hari non-trading dengan harga penutupan dari tanggal sebelumnya yang tersedia. Kami menerapkan langkah ini karena aset yang dipilih berasal dari berbagai pasar dengan hari trading yang berbeda-beda, sehingga penting untuk menstandarisasi dataset demi konsistensi.
![Visualisasi data historis aset](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. Definisikan masalah.

Tentukan spesifikasi masalah dengan mengonfigurasi parameter dalam dictionary `qubo_settings`.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. Definisikan pengaturan optimizer dan ansatz. (Opsional)
Secara opsional, tentukan persyaratan khusus untuk proses optimasi, termasuk pemilihan optimizer dan parameternya, serta spesifikasi primitive dan konfigurasinya.

Untuk Tailored Ansatz, ukuran populasi yang dipilih didasarkan pada eksperimen sebelumnya yang menunjukkan bahwa nilai ini menghasilkan optimasi yang stabil dan efisien.

Dalam kasus Real Amplitudes Ansatz, kamu bisa mengikuti hubungan linear antara ``population_size`` dan jumlah Qubit dalam Circuit. Sebagai aturan perkiraan, disarankan untuk menggunakan **minimum** ``population_size ~ 0.8 * n_qubits`` untuk ansatz `real_amplitudes`.

Diharapkan bahwa Optimized Real Amplitudes akan memiliki performa optimasi yang lebih baik daripada ansatz Real Amplitudes. Namun, jumlah variabel yang perlu dioptimalkan dalam ansatz ini meningkat jauh lebih cepat daripada pada kasus Real Amplitudes (lihat [naskah](https://arxiv.org/pdf/2412.19150)). Jadi, untuk masalah besar, Optimized Real Amplitudes memerlukan lebih banyak eksekusi Circuit. Optimized Real Amplitudes kemungkinan berguna untuk masalah yang memerlukan hingga 100 Qubit, tetapi disarankan untuk berhati-hati saat mengatur parameter ``population_size``. Sebagai contoh scale-up dalam ``population_size``, tabel sebelumnya menunjukkan bahwa untuk masalah 84 Qubit, Optimized Real Amplitudes memerlukan 120 ``population_size``, sementara untuk masalah 56 Qubit, ``population_size`` sebesar 40 sudah cukup.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

Juga mungkin untuk memilih ansatz tertentu. Berikut menggunakan ansatz ``'Tailored'``.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. Jalankan masalah.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. Ambil hasil.
Seperti yang disebutkan di bagian [Output](#output), fungsi ini mengembalikan dictionary dengan trajektori investasi yang diurutkan dari terendah ke tertinggi berdasarkan nilai fungsi objektifnya. Kumpulan hasil ini memungkinkan identifikasi trajektori dengan biaya terendah dan evaluasi investasi yang sesuai. Selain itu, ini menyediakan analisis berbagai trajektori, memfasilitasi pemilihan yang paling sesuai dengan kebutuhan atau tujuan tertentu. Fleksibilitas ini memastikan bahwa pilihan bisa disesuaikan dengan berbagai preferensi atau skenario.
Mulailah dengan menyajikan strategi hasil yang mencapai biaya objektif terendah yang ditemukan selama proses.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>